# RTOFS virtual Icechunk: smoke test

Small, rerunnable end-to-end test of `rtofs_build.py` against a **temporary local repository**.
It covers runs on both sides of every transition found in the source audit: model version
93.0 → 93.1 (2025-07-28/29), 5 → 8 lead days (2025-09-17/18), the 4-lead-day runs (2026-08-24),
a run containing two unreadable files (2025-07-29), and the newest run.

In [1]:
import sys, time, platform, tempfile, shutil, urllib.request
from pathlib import Path
import numpy as np, xarray as xr, zarr, icechunk as ic, virtualizarr, obstore, netCDF4
sys.path.insert(0, str(Path.cwd()))
import rtofs_build as rb
print("python", platform.python_version())
for m in (ic, virtualizarr, xr, zarr, obstore, netCDF4, np):
    print(f"{m.__name__:12s} {m.__version__}")

python 3.13.18
icechunk     2.3.1
virtualizarr 2.8.0
xarray       2026.9.0
zarr         3.4.2
obstore      0.12.1
netCDF4      1.7.5
numpy        2.5.5


## 1. Discover (real method, then pick a small subset)

In [2]:
files, other = rb.discover()
rb.check_listing(files)
print(len(files), "files;", "unmatched entries:", other)
RUNS = ["2025-07-28", "2025-07-29", "2025-09-17", "2025-09-18", "2026-08-24"]
newest = max(f.run for f in files)
runs = [np.datetime64(r, "ns") for r in RUNS] + [newest]
sub = [f for f in files if f.run in runs]
print(len(sub), "files in", len(runs), "runs:", [str(r)[:10] for r in runs])

3682 files; unmatched entries: []
70 files in 6 runs: ['2025-07-28', '2025-07-29', '2025-09-17', '2025-09-18', '2026-08-24', '2026-11-17']


## 2. Byte-range access (no credentials: anonymous HTTPS)

In [3]:
req = urllib.request.Request(sub[-1].url, headers={"Range": "bytes=0-7"})
with urllib.request.urlopen(req) as r:
    assert r.status == 206, r.status
    magic = r.read()
assert magic == b"\x89HDF\r\n\x1a\n", magic
print("HTTP 206, HDF5 magic OK")

HTTP 206, HDF5 magic OK


## 3. Inspect one file (virtual open: chunk index, MT, Depth only)

In [4]:
reg = rb.make_registry()
vds = rb.open_virtual(sub[-1].url, reg)
ma = vds[sub[-1].var].data
print(vds)
print("codecs:", ma.metadata.codecs, "\nchunks:", ma.metadata.chunk_grid.chunk_shape, "fill:", ma.metadata.fill_value)

<xarray.Dataset> Size: 2GB
Dimensions:    (MT: 1, Depth: 33, Y: 3298, X: 4500)
Coordinates:
  * MT         (MT) datetime64[ns] 8B 2026-11-25
    Date       (MT) float64 8B ManifestArray<shape=(1,), dtype=float64, chunk...
  * Depth      (Depth) float32 132B 0.0 10.0 20.0 30.0 ... 4.5e+03 5e+03 5.5e+03
    Y          (Y) int32 13kB ManifestArray<shape=(3298,), dtype=int32, chunk...
    X          (X) int32 18kB ManifestArray<shape=(4500,), dtype=int32, chunk...
    Latitude   (Y, X) float32 59MB ManifestArray<shape=(3298, 4500), dtype=fl...
    Longitude  (Y, X) float32 59MB ManifestArray<shape=(3298, 4500), dtype=fl...
Data variables:
    v          (MT, Depth, Y, X) float32 2GB ManifestArray<shape=(1, 33, 3298...
Attributes:
    Conventions:  CF-1.0
    title:        HYCOM ATLb2.00
    institution:  National Centers for Environmental Prediction
    source:       HYCOM archive file
    experiment:   93.1
    history:      archv2ncdf3z
codecs: (BytesCodec(endian='little'), Zlib(codec_na

## 4. Open the subset (cached per file; unreadable files are recorded, not fatal)

In [5]:
t0 = time.time()
refs, bad = rb.open_files(sub, workers=4)
print(f"{len(refs)} readable, {len(bad)} unreadable, {time.time()-t0:.0f}s")
for b in bad:
    print("UNREADABLE", b.src.path, "-", b.reason)
rb.check_compatible(refs)
T = rb.load_templates(refs, reg)

68 readable, 2 unreadable, 0s
UNREADABLE 2025/07/2025-07-29-00.2025-08-01-00.v.nc - Can't synchronously determine if attribute exists by name (incorrect metadata checksum after all read attempts)
UNREADABLE 2025/07/2025-07-29-00.2025-08-03-00.v.nc - Can't synchronously determine if attribute exists by name (incorrect metadata checksum after all read attempts)


## 5. Assemble both groups and write to a temporary repository

In [6]:
fc = rb.build_forecast(refs, T)
best = rb.build_best(refs, T)
tmp = tempfile.mkdtemp(prefix="rtofs_smoke_")
repo = rb.open_or_create(tmp)
snap = rb.write(repo, refs, T, "smoke test")
print("committed", snap, "to", tmp)

  2026-11-18T21:53:10.577684Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

  2026-11-18T21:53:10.579491Z  WARN icechunk_storage::readback: conditional PUT is enabled but `unsafe_use_metadata` is disabled — lost-response recovery for conditional writes requires user metadata to stamp write-ids; without it, transient PUT failures may surface as spurious conflicts even when the write actually landed. See icechunk_storage::Settings::unsafe_use_metadata.
    at icechunk-storage/src/readback.rs:31



committed ZWJ3R9ZES6F9T44SYAGG to /tmp/rtofs_smoke_it8gy_pr


## 6. Reopen from the consumer read path (fresh `Repository.open`, explicit HTTP authorization)

In [7]:
zarr.config.set({"async.concurrency": 32})  # modest: one university web server serves every read
t0 = time.time()
F = rb.open_readonly(tmp, "forecast", chunks={})
B = rb.open_readonly(tmp, "best", chunks={})
print(f"metadata open {time.time()-t0:.2f}s")
F

metadata open 0.09s


  2026-11-18T21:53:13.885325Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343

  2026-11-18T21:53:13.947994Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:343



<xarray.Dataset> Size: 188GB
Dimensions:                  (forecast_reference_time: 6, lead: 8, Depth: 33,
                              Y: 3298, X: 4500)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 48B 202...
    experiment               (forecast_reference_time) <U8 192B dask.array<chunksize=(6,), meta=np.ndarray>
    n_leads                  (forecast_reference_time) int8 6B dask.array<chunksize=(6,), meta=np.ndarray>
  * lead                     (lead) timedelta64[ns] 64B 1 days 2 days ... 8 days
    valid_time               (forecast_reference_time, lead) datetime64[ns] 384B dask.array<chunksize=(6, 8), meta=np.ndarray>
  * Depth                    (Depth) float32 132B 0.0 10.0 ... 5e+03 5.5e+03
    Latitude                 (Y, X) float32 59MB dask.array<chunksize=(1649, 2250), meta=np.ndarray>
    Longitude                (Y, X) float32 59MB dask.array<chunksize=(1649, 2250), meta=np.ndarray>
Dimensions without coordinates: Y, X
Data variables:
    u                        (forecast_reference_time, lead, Depth, Y, X) float32 94GB dask.array<chunksize=(1, 1, 3, 413, 563), meta=np.ndarray>
    v                        (forecast_reference_time, lead, Depth, Y, X) float32 94GB dask.array<chunksize=(1, 1, 3, 413, 563), meta=np.ndarray>
Attributes:
    Conventions:        CF-1.8
    title:              Global RTOFS (HYCOM 1/12 deg) ocean velocity, z-level...
    institution:        National Centers for Environmental Prediction
    source:             HYCOM ATLb2.00 (HYCOM archive file), experiments 93.0...
    source_archive:     https://hriai01.tamucc.edu/data/MASTR_MODELS/rtofs/hi...
    source_format:      NetCDF-4/HDF5, zlib level 1, referenced in place (vir...
    layout:             forecast
    source_file_count:  68
    metadata_repairs:   Longitude wrapped from the source's 74..794 (modulo 3...
    grid_sha256:        a81a01af6e53f1a1720be00826ee533d7886ea272e89ba73711c1...

## 7. Structure

In [8]:
assert dict(F.sizes) == {"forecast_reference_time": 6, "lead": 8, "Depth": 33, "Y": 3298, "X": 4500}
assert list(F.n_leads.values) == [5, 5, 5, 8, 4, 8]
assert list(F.experiment.values) == ["93.0"] + ["93.1"] * 5
assert F.u.encoding["chunks"] == (1, 1, 3, 413, 563) and F.u.dtype == np.float32
assert (np.diff(B.time.values) == np.timedelta64(1, "D")).all()
assert -180 <= float(F.Longitude.min()) and float(F.Longitude.max()) < 180
assert (F.valid_time.values == F.forecast_reference_time.values[:, None] + F.lead.values[None, :]).all()
print("structure OK;", B.attrs["missing_days"], "days in /best have no run in this subset")

structure OK; 459 days in /best have no run in this subset


## 8. Real data: store vs source, padding, unreadable slots, best fallback

In this subset run 2025-07-30 is absent, so /best has no fallback for 2025-08-03 (NaT, lead -1). In the
full build that day comes from run 2025-07-30, and /best loses nothing to the unreadable files.

In [9]:
win = dict(Depth=slice(0, 3), Y=slice(1500, 1700), X=slice(1000, 1300))  # western tropical Pacific, 0-15N 154-178E
def src(f, var):
    with netCDF4.Dataset(f.url + "#mode=bytes") as d:
        return d[var][0, 0:3, 1500:1700, 1000:1300].filled(np.nan)
for f in [sub[0], sub[-1]] + [r.src for r in refs if r.src.run == np.datetime64("2026-08-24", "ns")][:1]:
    t0 = time.time()
    a = F[f.var].sel(forecast_reference_time=f.run, lead=np.timedelta64(f.lead, "D")).isel(**win).values
    np.testing.assert_array_equal(a, src(f, f.var))
    print(f"{f.name}: identical to source ({time.time()-t0:.2f}s, {np.isfinite(a).mean():.0%} ocean)")

pad = F.u.sel(forecast_reference_time="2025-07-28").isel(lead=7, **win).values
assert np.isnan(pad).all(); print("lead 8 of a 5-day run: all NaN")
for b in bad:
    x = F[b.src.var].sel(forecast_reference_time=b.src.run, lead=np.timedelta64(b.src.lead, "D")).isel(**win).values
    assert np.isnan(x).all()
    o = B.sel(time=b.src.valid)
    assert o.forecast_reference_time.values != b.src.run
    print(f"unreadable {b.src.name}: NaN in /forecast; /best {str(b.src.valid)[:10]} comes from run "
          f"{str(o.forecast_reference_time.values)[:10]} lead {o.lead.values.astype('timedelta64[D]')}")
last = B.isel(time=slice(-1, None))
np.testing.assert_array_equal(last.v.isel(**win).values[0],
    F.v.sel(forecast_reference_time=newest).isel(lead=-1, **win).values)
print("/best newest day == newest run, lead 8")

2025-07-28-00.2025-07-29-00.u.nc: identical to source (7.76s, 100% ocean)


2026-11-17-00.2026-11-25-00.v.nc: identical to source (4.84s, 100% ocean)


2026-08-24-00.2026-08-25-00.u.nc: identical to source (4.73s, 100% ocean)
lead 8 of a 5-day run: all NaN
unreadable 2025-07-29-00.2025-08-01-00.v.nc: NaN in /forecast; /best 2025-08-01 comes from run 2025-07-28 lead 4 days
unreadable 2025-07-29-00.2025-08-03-00.v.nc: NaN in /forecast; /best 2025-08-03 comes from run NaT lead -1 days


/best newest day == newest run, lead 8


## 9. Timings (concurrency 32, local repository, source at TAMU-CC over the internet)

In [10]:
def timed(label, fn):
    t0 = time.time(); fn(); print(f"{label:45s} {time.time()-t0:6.2f}s")
d = B.time.values[-1]
timed("one native chunk (3 x 413 x 563)", lambda: B.u.sel(time=d).isel(Depth=slice(0, 3), Y=slice(0, 413), X=slice(0, 563)).values)
timed("global surface map, one variable (88 chunks)", lambda: B.u.sel(time=d).isel(Depth=0).values)
timed("Gulf of Mexico 0-20 m, u+v, one day", lambda: B[["u", "v"]].sel(time=d).isel(Depth=slice(0, 3), Y=slice(1733, 1912), X=slice(2349, 2573)).load())
timed("point series, surface, last 8 days", lambda: B.u.isel(time=slice(-8, None), Depth=slice(0, 1), Y=slice(1800, 1801), X=slice(2450, 2451)).values)

one native chunk (3 x 413 x 563)                0.41s


global surface map, one variable (88 chunks)    2.02s


Gulf of Mexico 0-20 m, u+v, one day             0.59s


point series, surface, last 8 days              0.46s


## 10. What is still unknown

- Whether the two unreadable files can be repaired at TAMU (they are excluded, and cached as
  unreadable until their modification time changes).
- Whether the files are daily means or 00 UTC snapshots; the metadata does not say.
- How the full 3,666-file build performs (manifest size, open time) — measured by the production build.
- Browser access: not tested; the source host sends no CORS header (TXLA audit), so browser readers
  would not see data.

## 11. Clean up the temporary repository only

In [11]:
shutil.rmtree(tmp)
print("removed", tmp)

removed /tmp/rtofs_smoke_it8gy_pr
